# RAG Inference Engine and Persistent Chatbot

This notebook is now organized like a small RAG application instead of a pile of helper functions. The same behavior is kept, but the responsibilities are grouped into three modules:

1. `ChromaKnowledgeBase`: reconnects to ChromaDB, rebuilds the LlamaIndex vector index, rebuilds BM25, and creates the hybrid retriever.
2. `JsonChatSessionStore`: saves and loads chat memory as one JSON file per chat in the `session` folder.
3. `RagNewsChatbot`: exposes the product-facing actions: ask one question, open a chat, continue a chat, inspect history, and show sources.

```mermaid
flowchart LR
    U[User question] --> APP[RagNewsChatbot]
    APP --> HR[HybridRetriever]
    HR --> VS[Semantic search\nChroma vector index]
    HR --> BM[Keyword search\nBM25]
    VS --> RRF[Reciprocal rank fusion]
    BM --> RRF
    RRF --> LLM[Gemma 3 1B via Ollama]
    LLM --> A[Answer + sources]
    APP <--> MEM[ChatMemoryBuffer]
    MEM <--> JSON[session/*.json]
```

The important product idea is separation of concerns: retrieval, generation, and persistence are related, but they should not be tangled together.

In [ ]:
# Run this once if the LlamaIndex integrations are missing, then restart the kernel.
%pip install llama-index-vector-stores-chroma llama-index-retrievers-bm25 llama-index-llms-ollama

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent
    print('project_root: ', project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag.inference.inference_observability import (
    DEFAULT_PHOENIX_ENDPOINT,
    DEFAULT_PHOENIX_PROJECT_NAME,
    PHOENIX_INSTALL_COMMAND,
    setup_phoenix_observability,
    trace_chat_session,
)
from src.rag.inference import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBED_MODEL_NAME,
    DEFAULT_HUGGINGFACE_MODEL_KEY,
    DEFAULT_LLM_PROVIDER,
    DEFAULT_OLLAMA_MODEL,
    HUGGINGFACE_CHAT_MODELS,
    create_rag_app,
    default_paths,
    print_sources,
    print_response,
)

project_root:  C:\Program Files\Studying\coding\RAG_project
resource module not available on Windows


In [2]:
PROJECT_ROOT = project_root
paths = default_paths(PROJECT_ROOT)
CHROMA_DIR = paths.chroma_dir
SESSION_DIR = paths.session_dir

COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBED_MODEL_NAME = DEFAULT_EMBED_MODEL_NAME
LLM_PROVIDER = DEFAULT_LLM_PROVIDER  # Choose: LLM_PROVIDER_OLLAMA or LLM_PROVIDER_HUGGINGFACE
OLLAMA_MODEL = DEFAULT_OLLAMA_MODEL
HUGGINGFACE_MODEL = DEFAULT_HUGGINGFACE_MODEL_KEY  # deepseek_v3, minimax_m3, qwen_3_5, or any full HF model id
HUGGINGFACE_PROVIDER = "auto"
PHOENIX_PROJECT_NAME = DEFAULT_PHOENIX_PROJECT_NAME
PHOENIX_COLLECTOR_ENDPOINT = DEFAULT_PHOENIX_ENDPOINT
LAUNCH_PHOENIX_SERVER = False  # Recommended: run `phoenix serve` in a terminal, then keep this False.
PHOENIX_SERVER_COMMAND = "phoenix serve"

print(f"Project root: {PROJECT_ROOT}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Session folder: {SESSION_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"Ollama model: {OLLAMA_MODEL}")
print(f"Hugging Face model: {HUGGINGFACE_CHAT_MODELS.get(HUGGINGFACE_MODEL, HUGGINGFACE_MODEL)}")
print(f"Phoenix project: {PHOENIX_PROJECT_NAME}")
print(f"Phoenix collector endpoint: {PHOENIX_COLLECTOR_ENDPOINT}")
print(f"Phoenix server command: {PHOENIX_SERVER_COMMAND}")

Project root: C:\Program Files\Studying\coding\RAG_project
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Session folder: C:\Program Files\Studying\coding\RAG_project\session
Collection name: run_testing
Embedding model: BAAI/bge-small-en-v1.5
LLM provider: ollama
Ollama model: gemma3:1b
Hugging Face model: deepseek-ai/DeepSeek-V3-0324
Phoenix project: rag-news-chatbot
Phoenix collector endpoint: http://localhost:6006
Phoenix server command: phoenix serve


In [ ]:
# Retrieval, fusion, and knowledge-base classes are now centralized in src/rag/.
# This notebook imports and uses the same implementation as app.py.

In [ ]:
# Session persistence and chatbot facade now come from src/rag/.
# This removes duplicate maintenance between the notebook and app.py.

## Build the Application Objects

Run the next cell once per notebook session. It creates:

- `knowledge_base`: reconnects to the persisted `news_chat` ChromaDB collection
- `session_store`: reads and writes chat JSON files under `session/`
- `rag_app`: the single object you use for query, chat, sources, and history

This is the main readability improvement: examples below call methods on `rag_app` instead of manually coordinating many functions and global variables.

## Run the RAG App

The next cells demonstrate the same functionality as before, but through one object: `rag_app`.

Use `rag_app.ask(...)` for one-off questions. Use `rag_app.open_chat(...)` and `rag_app.chat(...)` for multi-turn conversations with persistent memory.

### Initialize the App

In [3]:
# Run this once after reopening the notebook.
# For stability on Windows, start Phoenix in a terminal with `phoenix serve`
# and keep LAUNCH_PHOENIX_SERVER = False in the config cell above.
# Phoenix must be configured before create_rag_app so LlamaIndex query/chat spans are captured.
phoenix_status = setup_phoenix_observability(
    project_name=PHOENIX_PROJECT_NAME,
    endpoint=PHOENIX_COLLECTOR_ENDPOINT,
    launch_server=LAUNCH_PHOENIX_SERVER,
    batch=True,
    raise_on_missing=False,
)
print(phoenix_status.message)
print(f"Phoenix UI: {phoenix_status.ui_url}")
print(f"Phoenix collector endpoint: {phoenix_status.endpoint}")
if not phoenix_status.enabled:
    print(f"Install Phoenix packages first: {PHOENIX_INSTALL_COMMAND}")
    print(f"Then start server in terminal: {PHOENIX_SERVER_COMMAND}")

# create_rag_app(...) is the main factory for this project.
# It reconnects to the stored Chroma vectors, configures the embedding model, configures the LLM,
# builds the hybrid retriever, and prepares JSON chat-session persistence.
# Output: rag_app becomes the one high-level object you use for ask(...), open_chat(...), and chat(...).
rag_app = create_rag_app(
    chroma_dir=CHROMA_DIR,
    session_dir=SESSION_DIR,
    collection_name=COLLECTION_NAME,
    embed_model_name=EMBED_MODEL_NAME,
    llm_provider=LLM_PROVIDER,
    ollama_model=OLLAMA_MODEL,
    huggingface_model=HUGGINGFACE_MODEL,
    huggingface_provider=HUGGINGFACE_PROVIDER,
    final_top_k=5,
)
knowledge_base = rag_app.knowledge_base

def notebook_chat(chat_id: str, message: str):
    """Run one notebook chat turn inside a saved chat session.

    Purpose:
    - Attach the saved chat id and notebook label to Phoenix spans created during this call.
    - Forward the actual answer generation to rag_app.chat(...), which uses LlamaIndex chat memory,
      retrieval from Chroma, and the chosen LLM.

    Output:
    - Returns the LlamaIndex chat response object for this user message.
    - If Phoenix is enabled, the underlying LlamaIndex work is also logged as spans.
    """
    # trace_chat_session(...) does not create a visible parent span by itself.
    # It adds metadata so child spans know this call came from the notebook interface.
    with trace_chat_session(chat_id, metadata={"interface": "notebook"}):
        return rag_app.chat(chat_id, message)

print(f"Stored vector count: {knowledge_base.count()}")
print("RAG app is ready.")

c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


OpenTelemetry Tracing Details
|  Phoenix Project: rag-news-chatbot
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix tracing is enabled. Run RAG queries, then open the Phoenix UI to inspect traces.
Phoenix UI: http://localhost:6006
Phoenix collector endpoint: http://localhost:6006/v1/traces
Stored vector count: 1621
RAG app is ready.


### Single-turn RAG: use this when the question does not need chat history.

In [ ]:
sample_query = "What model you are ? and what is the latest information you can provide to me ?"
query_response = rag_app.ask(sample_query)

print_response(query_response)
print("-" * 70)
print_sources(query_response)

### Multi-turn RAG With Saved Chat History

`rag_app.open_chat(...)` loads the saved JSON file when it exists, or creates a new one when it does not. Every `rag_app.chat(...)` call saves the updated conversation automatically.

In [ ]:
# Create or load one persistent chat session.
# If session/Sanae_Takaichi.json exists, this restores its previous messages.
chat_id = rag_app.open_chat("Test_Chat_session")

chat_response = notebook_chat(
    chat_id=chat_id,
    message="Who is donald trump ?",
)

print_response(chat_response)
print('-' * 70)
print_sources(chat_response)

In [ ]:
# Follow-up question: this uses the same saved chat memory, so "she" refers to Sanae Takaichi.
follow_up_response = notebook_chat(
    chat_id=chat_id,
    message="Did he met with Xi ?",
)
print_response(follow_up_response)
print('-' * 70)
print_sources(follow_up_response)

In [ ]:
# Follow-up question: this uses the same saved chat memory, so "she" refers to Sanae Takaichi.
follow_up_response = notebook_chat(
    chat_id=chat_id,
    message="What is the relationship between Donald Trump and Stephen Kevin Steve Bannon ?",
)
print_response(follow_up_response)
print('-' * 70)
print_sources(follow_up_response)

In [ ]:
# Inspect the current in-memory history for this chat.
rag_app.show_history(chat_id)

In [ ]:
# List saved chat files on disk.
rag_app.list_saved_chats()

### Reopen an Existing Chat

After reopening the notebook, run the import/config/module cells and the app initialization cell first. Then `rag_app.open_chat("Sanae Takaichi")` restores the saved conversation from `session/Sanae_Takaichi.json`.

In [4]:
loaded_chat_id = rag_app.open_chat("Test_Chat_session")

In [5]:
rag_app.show_history(loaded_chat_id)

user: Who is donald trump ?
assistant: Donald Trump is an American businessman and politician who served as the 45th President of the United States from 2017 to 2021. He was a real estate developer, television personality, and author before entering politics.

Here's a breakdown of key points about him:

*   **Political Career:** Before entering politics, Trump built a successful real estate business, focusing on luxury homes and commercial properties. He was also a host on the television show "The Apprentice."
*   **Political Career:** He ran for president in 2016 and won against Hillary Clinton, campaigning on a populist platform centered on nationalism, immigration control, and economic nationalism.
*   **Presidency:** As President, he oversaw significant policy changes, including tax cuts, deregulation, and efforts to challenge international agreements. He also pursued a more assertive foreign policy, including withdrawing from the Iran nuclear deal and imposing tariffs on trade wi

In [6]:
# Continue a loaded chat without rerunning the original conversation cells.
reloaded_response = notebook_chat(
    chat_id=loaded_chat_id,
    message="Did Trump hate Hong Kong ?",
)

print_response(reloaded_response)
print('-' * 70)
print_sources(reloaded_response)

That’s a really complex and widely debated question. While Trump didn’t explicitly declare a hatred for Hong Kong, his rhetoric and actions consistently painted a picture of deep distrust and concern regarding the city’s relationship with China.

Here’s a more nuanced breakdown:

*   **Initial Sentiment:** Early in his presidency, he voiced admiration for Hong Kong’s legal system and its position as a global center. He frequently talked about the city’s role in the world.
*   **Growing Criticism:** As his administration became more focused on countering China's influence, his criticisms intensified. He made statements suggesting the city was a “disaster” and that the government was abusing its power.
*   **"China" as a Threat:** Trump frequently framed the situation in Hong Kong as a direct threat to American interests, linking it to the broader geopolitical struggle between the US and China.

**Ultimately, many experts agree he held a deep-seated anxiety and suspicion about the trajec